# Data cleaning — Big 5 leagues

This notebook is the **only** script for data cleaning: it runs all pipeline steps in one place. There are no separate `.py` files to run; all logic (including column rename maps) is in the cells below.

**How to use:** Set the notebook’s working directory to `Code/DataCleaning/`, then **Run All**. The output of each cell will stay visible under the cell. Run the cells in order from top to bottom. Paths resolve to the project root for `Databases/` and to this folder for `outputs/`.

---

## Pipeline overview

| Step | What it does |
|------|----------------|
| **1. Encoding fix** | Fixes broken characters (mojibake) in Player and Team names in the performance Excels; normalises Player names to NFC; exports a match-rate audit to `outputs/`. |
| **2. Columns & Comp** | Renames long Excel column headers to short names; cleans Nation and Comp; drops `source_file`; applies Nation/Comp cleaning to salary and team-wage files. |
| **3. Drop empty wages** | Removes rows in the player salary CSVs where **Weekly Wages** or **Annual Wages** is missing. |
| **4. Standardize team names (salaries)** | Maps salary Team names to match performance/team wages; removes "N Players" rows; keeps one row per player (higher wage) for transfer duplicates; verifies team names. |
| **5. NFC and merge key** | Applies clean_player_name() to Player in all 6 files; match-rate diagnostic; exports player_name_audit.csv. |
| **6. Merge** | Merges player salaries and team wages into the 3 performance files; saves to `data/processed/merged/` (*_merged.xlsx, all rows). |
| **7. Cleanup & split** | Drops duplicates, all-null and Rk/team-wage columns; overwrites `data/processed/merged/`; writes **Analysis_Ready** (matched, Min≥90) to `data/processed/feature_engineered/` and **Unmatched** to `data/processed/unmatched/`. |
| **8. Validation report** | Runs a 10-check quality gate on the Analysis_Ready files. |

---

## Folder layout (project root)

- **`data/raw/performance_stats/`** — Performance Excel files (Defenders, Midfielders_Forwards, Goalkeepers); cleaned in place.
- **`data/raw/player_salaries/`** — Player salary CSVs (2022–23 to 2024–25); cleaned in place.
- **`data/raw/club_wages/`** — Team total wage CSVs; cleaned in place.
- **`data/processed/merged/`** — Merged outputs (all rows): *_merged.xlsx.
- **`data/processed/feature_engineered/`** — Analysis-ready files (matched, Min≥90): *_analysis_ready.xlsx. Input for feature engineering.
- **`data/processed/unmatched/`** — Rows with Salary_Matched=False (kept for future use).
- **`outputs/`** — Generated artifacts (e.g. player_name_audit.csv).

In [ ]:
from pathlib import Path
import unicodedata
import pandas as pd

# Run this notebook from Code/DataCleaning/. Project root = parent of Code/.
BASE = Path(".").resolve()  # Code/DataCleaning
PROJECT_ROOT = BASE.parent.parent  # TFG_Código (Database lives here)
DATABASES = PROJECT_ROOT / "Database"
PERFORMANCE_DIR = DATABASES / "Performance_Stats"
SALARIES_DIR = DATABASES / "Player_Salaries"
TEAM_WAGES_DIR = DATABASES / "Club_Total_Wages"
OUTPUTS_DIR = BASE / "outputs"
MERGED_DIR = DATABASES / "Merged_Player_Data"
ANALYSIS_READY_DIR = DATABASES / "Player_Data_Ready"
UNMATCHED_DIR = DATABASES / "Unmatched"

for d in (PERFORMANCE_DIR, SALARIES_DIR, TEAM_WAGES_DIR, OUTPUTS_DIR, MERGED_DIR, ANALYSIS_READY_DIR, UNMATCHED_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Performance Excel filenames (as stored in data/raw/performance_stats/)
PERF_FILENAMES = [
    "defenders_combined.xlsx",
    "midfielders_forwards_combined.xlsx",
    "goalkeepers_combined.xlsx",
]
# Salary and team wage CSVs (as stored in Database/Player_Salaries/ and Database/Club_Total_Wages/)
SALARY_FILENAMES = [
    "Big5_2022-2023_player_salaries.csv",
    "Big5_2023-2024_player_salaries.csv",
    "Big5_2024-2025_player_salaries.csv",
]
TEAM_WAGES_FILENAMES = [
    "Big5_2022-2023_team_total_wages.csv",
    "Big5_2023-2024_team_total_wages.csv",
    "Big5_2024-2025_team_total_wages.csv",
]
SEASONS = ["2022-2023", "2023-2024", "2024-2025"]

In [ ]:
# Column rename maps for each performance file (FBRef Excel long headers -> short names).
# Defenders, Midfielders_Forwards, Goalkeepers each have different column layouts.

DEFENDERS_RENAME = {
    "Unnamed: 0_level_0_Rk": "Rk", "Unnamed: 1_level_0_Player": "Player", "Unnamed: 2_level_0_Int": "Int",
    "Unnamed: 3_level_0_Sh": "Sh", "Unnamed: 4_level_0_Mn/MP": "Mn/MP", "Unnamed: 5_level_0_PPM": "PPM",
    "Unnamed: 6_level_0_CrdY": "CrdY", "Unnamed: 7_level_0_OG": "OG", "Unnamed: 8_level_0_L": "L",
    "Unnamed: 9_level_0_Season": "Season", "Unnamed: 10_level_0_Age": "Age", "Unnamed: 11_level_0_Nation": "Nation",
    "Unnamed: 12_level_0_Team": "Team", "Unnamed: 13_level_0_Comp": "Comp",
    "Playing Time_MP": "MP", "Playing Time_Min": "Min", "Playing Time_90s": "90s", "Playing Time_Starts": "Starts",
    "Playing Time_Subs": "Subs", "Playing Time_unSub": "unSub",
    "Performance_Gls": "Gls", "Performance_Ast": "Ast", "Performance_G+A": "G+A", "Performance_G-PK": "G-PK",
    "Performance_PK": "PK", "Performance_PKatt": "PKatt", "Performance_PKm": "PKm",
    "Standard_Sh": "Sh_Standard", "Standard_G/Sh": "G/Sh", "Standard_G/SoT": "G/SoT", "Standard_SoT": "SoT", "Standard_SoT%": "SoT%",
    "Unnamed: 32_level_0_TklW": "TklW", "Unnamed: 33_level_0_Int": "Int_Def", "Unnamed: 34_level_0_Tkl+Int": "Tkl+Int",
    "Playing Time_Mn/MP": "Mn/MP_Detail", "Playing Time_Min%": "Min%", "Playing Time_Mn/Start": "Mn/Start",
    "Playing Time_Compl": "Compl", "Playing Time_Mn/Sub": "Mn/Sub",
    "Team Success_PPM": "PPM_Team", "Team Success_onG": "onG", "Team Success_onGA": "onGA", "Team Success_+/-": "+/-", "Team Success_On-Off": "On-Off",
    "Performance_CrdY": "CrdY_Detail", "Performance_CrdR": "CrdR", "Performance_2CrdY": "2CrdY",
    "Performance_Fls": "Fls", "Performance_Fld": "Fld", "Performance_Off": "Off", "Performance_PKwon": "PKwon", "Performance_PKcon": "PKcon",
    "Performance_OG": "OG_Detail", "Performance_GA": "GA", "Performance_SoTA": "SoTA", "Performance_Saves": "Saves", "Performance_Save%": "Save%",
    "Performance_W": "W", "Performance_D": "D", "Performance_L": "L_Detail", "Performance_CS": "CS", "Performance_CS%": "CS%",
    "Unnamed: 63_level_0_Pos": "Pos",
}
MIDFIELDERS_FORWARDS_RENAME = {
    "Unnamed: 0_level_0_Rk": "Rk", "Unnamed: 1_level_0_Player": "Player", "Unnamed: 2_level_0_Sh": "Sh", "Unnamed: 3_level_0_Int": "Int",
    "Unnamed: 4_level_0_Mn/MP": "Mn/MP", "Unnamed: 5_level_0_PPM": "PPM", "Unnamed: 6_level_0_CrdY": "CrdY", "Unnamed: 7_level_0_Off": "Off",
    "Unnamed: 8_level_0_Season": "Season", "Unnamed: 9_level_0_Age": "Age", "Unnamed: 10_level_0_Nation": "Nation",
    "Unnamed: 11_level_0_Team": "Team", "Unnamed: 12_level_0_Comp": "Comp",
    "Playing Time_MP": "MP", "Playing Time_Min": "Min", "Playing Time_90s": "90s", "Playing Time_Starts": "Starts",
    "Playing Time_Subs": "Subs", "Playing Time_unSub": "unSub",
    "Performance_Gls": "Gls", "Performance_Ast": "Ast", "Performance_G+A": "G+A", "Performance_G-PK": "G-PK",
    "Performance_PK": "PK", "Performance_PKatt": "PKatt", "Performance_PKm": "PKm",
    "Standard_Sh": "Sh_Standard", "Standard_G/Sh": "G/Sh", "Standard_G/SoT": "G/SoT", "Standard_SoT": "SoT", "Standard_SoT%": "SoT%",
    "Unnamed: 31_level_0_TklW": "TklW", "Unnamed: 32_level_0_Int": "Int_Def", "Unnamed: 33_level_0_Tkl+Int": "Tkl+Int",
    "Playing Time_Mn/MP": "Mn/MP_Detail", "Playing Time_Min%": "Min%", "Playing Time_Mn/Start": "Mn/Start",
    "Playing Time_Compl": "Compl", "Playing Time_Mn/Sub": "Mn/Sub",
    "Team Success_PPM": "PPM_Team", "Team Success_onG": "onG", "Team Success_onGA": "onGA", "Team Success_+/-": "+/-", "Team Success_On-Off": "On-Off",
    "Performance_CrdY": "CrdY_Detail", "Performance_CrdR": "CrdR", "Performance_2CrdY": "2CrdY",
    "Performance_Fls": "Fls", "Performance_Fld": "Fld", "Performance_Off": "Off_Detail", "Performance_PKwon": "PKwon", "Performance_PKcon": "PKcon",
    "Performance_OG": "OG", "Unnamed: 53_level_0_Pos": "Pos",
}
GOALKEEPERS_RENAME = {
    "Unnamed: 0_level_0_Rk": "Rk", "Unnamed: 1_level_0_Player": "Player", "Unnamed: 2_level_0_Int": "Int",
    "Unnamed: 3_level_0_Mn/MP": "Mn/MP", "Unnamed: 4_level_0_PPM": "PPM", "Unnamed: 5_level_0_CrdY": "CrdY", "Unnamed: 6_level_0_OG": "OG",
    "Unnamed: 7_level_0_GA": "GA_Summary", "Unnamed: 8_level_0_PKvs": "PKvs_Summary",
    "Unnamed: 9_level_0_Season": "Season", "Unnamed: 10_level_0_Age": "Age", "Unnamed: 11_level_0_Nation": "Nation",
    "Unnamed: 12_level_0_Team": "Team", "Unnamed: 13_level_0_Comp": "Comp",
    "Playing Time_MP": "MP", "Playing Time_Min": "Min", "Playing Time_90s": "90s", "Playing Time_Starts": "Starts",
    "Playing Time_Subs": "Subs", "Playing Time_unSub": "unSub",
    "Performance_Gls": "Gls", "Performance_Ast": "Ast", "Performance_G+A": "G+A", "Performance_G-PK": "G-PK",
    "Performance_PK": "PK", "Performance_PKatt": "PKatt", "Performance_PKm": "PKm",
    "Unnamed: 27_level_0_TklW": "TklW", "Unnamed: 28_level_0_Int": "Int_Def", "Unnamed: 29_level_0_Tkl+Int": "Tkl+Int",
    "Playing Time_Mn/MP": "Mn/MP_Detail", "Playing Time_Min%": "Min%", "Playing Time_Mn/Start": "Mn/Start",
    "Playing Time_Compl": "Compl", "Playing Time_Mn/Sub": "Mn/Sub",
    "Team Success_PPM": "PPM_Team", "Team Success_onG": "onG", "Team Success_onGA": "onGA", "Team Success_+/-": "+/-", "Team Success_On-Off": "On-Off",
    "Performance_CrdY": "CrdY_Detail", "Performance_CrdR": "CrdR", "Performance_2CrdY": "2CrdY",
    "Performance_Fls": "Fls", "Performance_Fld": "Fld", "Performance_Off": "Off", "Performance_PKwon": "PKwon", "Performance_PKcon": "PKcon",
    "Performance_OG": "OG_Detail", "Performance_GA": "GA", "Performance_SoTA": "SoTA", "Performance_Saves": "Saves", "Performance_Save%": "Save%",
    "Performance_W": "W", "Performance_D": "D", "Performance_L": "L", "Performance_CS": "CS", "Performance_CS%": "CS%", "Performance_PKvs": "PKvs",
    "Penalty Kicks_PKA": "PKA", "Penalty Kicks_PKsv": "PKsv", "Penalty Kicks_PKm": "PKm_GK", "Penalty Kicks_Save%": "Save%_PK",
    "Unnamed: 63_level_0_Pos": "Pos",
}

## Step 1 — Encoding fix (Player, Team) + NFC

**Why this step:** Performance Excel files sometimes contain **mojibake**: characters that were stored in one encoding (e.g. Latin-1) but read as another (e.g. UTF-8), so e.g. `é` appears as `Ã©`. We fix this by re-interpreting the bytes as Latin-1 and decoding as UTF-8. A small manual map corrects a few truncated names that cannot be recovered automatically.

**NFC (Unicode normalization):** We normalise all Player names to NFC so that the same character (e.g. `é`) is stored in a single, consistent form. That improves matching between performance data and salary CSVs (e.g. when joining on Player + Season + Team).

**Output:** Performance files and salary CSVs are updated in place; a **player_name_audit.csv** is written to `outputs/` with one row per Player–Season–File and a flag indicating whether that player was found in the corresponding salary CSV (useful to check match rates).

In [ ]:
TRUNCATED_NAME_MAP = {
    "AdriÃ Altimira": "Adrià Altimira",
    "AdriÃ Pedrosa": "Adrià Pedrosa",
    "JosÃ© Luis GayÃ": "José Luis Gayà",
    "Arnau SolÃ": "Arnau Solà",
}

def fix_encoding(s):
    if not isinstance(s, str):
        return s
    if s in TRUNCATED_NAME_MAP:
        return TRUNCATED_NAME_MAP[s]
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s

def normalize_name(s):
    if not isinstance(s, str):
        return s
    s = fix_encoding(s).strip()
    s = unicodedata.normalize("NFC", s)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.lower().strip()

PERF_CONFIG = [
    {"file": PERF_FILENAMES[0], "player": "Unnamed: 1_level_0_Player", "team": "Unnamed: 12_level_0_Team", "season": "Unnamed: 9_level_0_Season", "comp": "Unnamed: 13_level_0_Comp"},
    {"file": PERF_FILENAMES[1], "player": "Unnamed: 1_level_0_Player", "team": "Unnamed: 11_level_0_Team", "season": "Unnamed: 8_level_0_Season", "comp": "Unnamed: 12_level_0_Comp"},
    {"file": PERF_FILENAMES[2], "player": "Unnamed: 1_level_0_Player", "team": "Unnamed: 12_level_0_Team", "season": "Unnamed: 9_level_0_Season", "comp": "Unnamed: 13_level_0_Comp"},
]

season_to_salary_keys = {s: set() for s in SEASONS}
perf_dfs = []

for cfg in PERF_CONFIG:
    path = PERFORMANCE_DIR / cfg["file"]
    if not path.exists():
        print(f"Skip (not found): {path}")
        continue
    df = pd.read_excel(path)
    player_col, team_col = cfg["player"], cfg["team"]
    df[player_col] = df[player_col].apply(lambda s: fix_encoding(s) if isinstance(s, str) else s)
    df[team_col] = df[team_col].apply(lambda s: fix_encoding(s) if isinstance(s, str) else s)
    df[player_col] = df[player_col].apply(
        lambda s: unicodedata.normalize("NFC", s.strip()) if isinstance(s, str) and str(s).strip() else s
    )
    df["_raw_player"] = df[player_col].copy()
    df["_file"] = cfg["file"]
    perf_dfs.append((cfg, df))

salary_dfs_s1 = []
for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if "Player" in df.columns:
        df["Player"] = df["Player"].apply(
            lambda s: unicodedata.normalize("NFC", s.strip()) if isinstance(s, str) and str(s).strip() else s
        )
    season = next((s for s in SEASONS if s in fname), None)
    if season and "Player" in df.columns:
        for p in df["Player"].dropna():
            season_to_salary_keys[season].add((normalize_name(p), season))
    df.to_csv(path, index=False)
    salary_dfs_s1.append(df)

audit_rows = []
for cfg, df in perf_dfs:
    player_col, team_col, season_col = cfg["player"], cfg["team"], cfg["season"]
    for season in SEASONS:
        mask = df[season_col].astype(str).str.strip() == season
        for _, row in df.loc[mask].iterrows():
            p = row[player_col]
            if pd.isna(p) or str(p).strip() == "":
                continue
            key = (normalize_name(p), season)
            audit_rows.append({
                "Player_fixed": row[player_col], "Season": season, "Team": row[team_col],
                "File": cfg["file"], "In_Salary_CSV": key in season_to_salary_keys[season],
            })

for cfg, df in perf_dfs:
    out = df.drop(columns=["_raw_player", "_file"], errors="ignore")
    out.to_excel(PERFORMANCE_DIR / cfg["file"], index=False)
    print(f"Saved {cfg['file']}")

audit_df = pd.DataFrame(audit_rows).drop_duplicates(subset=["Player_fixed", "Season", "File"])
# NOTE: This audit uses PRE-CLEANING salary keys (before Steps 3–4). Match rates here differ from Step 5 (authoritative). Useful for encoding diagnostics only.
audit_df.to_csv(OUTPUTS_DIR / "player_name_audit_pre_clean.csv", index=False)
print(f"Exported player_name_audit_pre_clean.csv ({len(audit_df)} rows)")

## Step 2 — Columns & Comp

**Column renames:** The performance Excels use long multi-level headers (e.g. `Unnamed: 1_level_0_Player`, `Playing Time_MP`). We replace them with short, readable names (e.g. `Player`, `MP`) using the mapping tables defined in the cell above (DEFENDERS_RENAME, MIDFIELDERS_FORWARDS_RENAME, GOALKEEPERS_RENAME — one per file type).

**Nation:** Some cells contain a language prefix plus the code (e.g. `"fr FRA"`). We keep only the last token so that Nation is a consistent 3-letter code (e.g. `"FRA"`).

**Comp (competition):** League names sometimes include a prefix (e.g. `"de Bundesliga"`, `"en Premier League"`). We strip that so Comp is one of: `Bundesliga`, `La Liga`, `Ligue 1`, `Premier League`, `Serie A`.

**Other:** We drop the `source_file` column from performance tables if present. Team wage CSVs get only Comp cleaning; player salary CSVs get only Nation cleaning. All files are read from and written back to their subfolders.

In [ ]:
# Rename maps (DEFENDERS_RENAME, MIDFIELDERS_FORWARDS_RENAME, GOALKEEPERS_RENAME) are defined in an earlier cell.

def clean_nation(val):
    """Keep only the 3-letter code, e.g. 'fr FRA' -> 'FRA'."""
    if not isinstance(val, str):
        return val
    parts = val.strip().split()
    return parts[-1] if len(parts) >= 2 else val

def clean_comp(val):
    """Normalise competition name to league only, e.g. 'de Bundesliga' -> 'Bundesliga'."""
    if not isinstance(val, str):
        return val
    for league in ("Bundesliga", "La Liga", "Ligue 1", "Premier League", "Serie A"):
        if league in val:
            return league
    return val

PERF_FILES = [
    (PERF_FILENAMES[0], DEFENDERS_RENAME),
    (PERF_FILENAMES[1], MIDFIELDERS_FORWARDS_RENAME),
    (PERF_FILENAMES[2], GOALKEEPERS_RENAME),
]

# --- Performance files: rename, clean Nation & Comp, drop source_file ---
for fname, rename_map in PERF_FILES:
    path = PERFORMANCE_DIR / fname
    if not path.exists():
        print(f"Skip (not found): {path}")
        continue
    df = pd.read_excel(path)
    rename_subset = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_subset)
    if "Nation" in df.columns:
        df["Nation"] = df["Nation"].apply(clean_nation)
    if "Comp" in df.columns:
        df["Comp"] = df["Comp"].apply(clean_comp)
    if "source_file" in df.columns:
        df = df.drop(columns=["source_file"])
    df.to_excel(path, index=False)
    print(f"Saved {fname}")

# --- Team wages: clean Comp only ---
for fname in TEAM_WAGES_FILENAMES:
    path = TEAM_WAGES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if "Comp" in df.columns:
        df["Comp"] = df["Comp"].apply(clean_comp)
    df.to_csv(path, index=False)
    print(f"Saved {fname}")

# --- Player salaries: clean Nation only ---
for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if "Nation" in df.columns:
        df["Nation"] = df["Nation"].apply(clean_nation)
    df.to_csv(path, index=False)
    print(f"Saved {fname}")

print("Step 2 done.")

## Step 3 — Drop empty wages

**Why:** Some rows in the player salary CSVs have missing **Weekly Wages** or **Annual Wages**. Those records are not useful for salary-based analysis or for joining with performance data when we need wage values. We remove any row where either of these two columns is empty (including `NaN` or blank/whitespace).

**Effect:** The three salary CSVs in `salaries/` are overwritten with only rows that have both wage fields present. Row counts are printed so you can see how many were removed per file.

In [ ]:
def is_empty(val):
    """Treat NaN, empty string, 'nan', or Unicode whitespace-only as empty (hardens against scraped data)."""
    if pd.isna(val):
        return True
    # Strip all Unicode whitespace (Z* categories) and common invisible chars
    invisible = (0x00A0, 0x200B, 0x200C, 0x200D, 0xFEFF)
    s = "".join(c for c in str(val) if ord(c) not in invisible and not (c.isspace() if hasattr(c, 'isspace') else c == ' '))
    s = s.strip()
    return s == "" or s.lower() == "nan"

for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    before = len(df)
    mask_weekly = df["Weekly Wages"].apply(is_empty)
    mask_annual = df["Annual Wages"].apply(is_empty)
    drop = mask_weekly | mask_annual
    df = df[~drop]
    removed = before - len(df)
    df.to_csv(path, index=False)
    print(f"{fname}: {before} -> {len(df)} rows (removed {removed} with empty wages)")

print("Step 3 done.")

## Step 4 — Standardize team names (salary CSVs only)

**Why:** Performance and team wages files already use consistent team names (e.g. "Brighton", "Paris Saint-Germain", "Atlético Madrid"). The player salary CSVs use different forms (e.g. "Brighton and Hove Albion", "Paris Saint Germain", "Atletico Madrid"). We map the salary **Team** column to the standard names so joins across all 6 files match.

**Steps applied only to the 3 salary CSVs (performance and team_wages are unchanged):**
1. **Team mapping** — Replace 14 known salary names with the standard form (e.g. "Internazionale" → "Inter", "Monchengladbach" → "Gladbach").
2. **Remove "N Players" rows** — Drop aggregate rows where Player is like "33 Players" or "26 Players".
3. **Transfer duplicates** — If a player appears twice in the same season (e.g. moved mid-season), keep the row with the **higher** weekly wage in EUR; drop the other.

**Verify:** For each season we print the sorted unique team names from performance, salaries, and team_wages (excluding summary rows like "2 Teams") and check: (a) no old ASCII names remain, (b) standard accented names are present, (c) team counts match expectations (98, 96, 96).

In [ ]:
import re

SALARY_TEAM_MAP = {
    "Brighton and Hove Albion": "Brighton",
    "Internazionale": "Inter",
    "Manchester United": "Manchester Utd",
    "Paris Saint Germain": "Paris Saint-Germain",
    "Monchengladbach": "Gladbach",
    "Wolverhampton Wanderers": "Wolves",
    "Atletico Madrid": "Atlético Madrid",
    "Alaves": "Alavés",
    "Cadiz": "Cádiz",
    "Almeria": "Almería",
    "Koln": "Köln",
    "Leganes": "Leganés",
    "Saint Etienne": "Saint-Étienne",
}

def extract_eur(s):
    if not isinstance(s, str):
        return 0
    m = re.search(r"€\s*([\d,]+)", s)
    return int(m.group(1).replace(",", "")) if m else 0

# --- Step 4.1: Apply team mapping to salary CSVs only ---
salary_dfs = []
for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    if "Team" in df.columns:
        df["Team"] = df["Team"].replace(SALARY_TEAM_MAP)
    salary_dfs.append((fname, df))

# --- Step 4.2: Remove "N Players" summary rows ---
print("Removed 'N Players' rows:")
for i, (fname, df) in enumerate(salary_dfs):
    before = len(df)
    df = df[~df["Player"].str.match(r"^\d+\s+Players$", na=False)]
    removed = before - len(df)
    salary_dfs[i] = (fname, df)
    print(f"  {fname}: {removed}")

# --- Step 4.3: Keep one row per player (higher weekly wage) ---
print("\nRemoved transfer duplicate rows:")
for i, (fname, df) in enumerate(salary_dfs):
    before = len(df)
    df["_eur_weekly"] = df["Weekly Wages"].apply(extract_eur)
    df = df.sort_values(["Player", "_eur_weekly"], ascending=[True, False])
    df = df.drop_duplicates(subset=["Player"], keep="first")
    df = df.drop(columns=["_eur_weekly"])
    removed = before - len(df)
    salary_dfs[i] = (fname, df)
    print(f"  {fname}: {removed}")

# --- Save ---
for fname, df in salary_dfs:
    df.to_csv(SALARIES_DIR / fname, index=False)
    print(f"Saved {fname}")

# --- Step 4.4: Verify team names per season (exclude "N Teams" from performance) ---
OLD_ASCII = set(SALARY_TEAM_MAP.keys())
EXPECTED_COUNTS = {"2022-2023": 98, "2023-2024": 96, "2024-2025": 96}

for idx, season in enumerate(SEASONS):
    perf_teams = set()
    for pname in PERF_FILENAMES:
        pdf = pd.read_excel(PERFORMANCE_DIR / pname)
        if "Team" not in pdf.columns or "Season" not in pdf.columns:
            continue
        mask = pdf["Season"].astype(str).str.strip() == season
        t = pdf.loc[mask, "Team"].dropna().astype(str).str.strip()
        perf_teams |= {x for x in t.unique() if not re.match(r"^\d+\s+Teams$", x)}
    sal_teams = set(salary_dfs[idx][1]["Team"].dropna().astype(str).str.strip().unique())
    tw_path = TEAM_WAGES_DIR / TEAM_WAGES_FILENAMES[idx]
    tw_teams = set()
    if tw_path.exists():
        tw_df = pd.read_csv(tw_path)
        # Raw team_wages CSVs use 'Squad'; Step 6 renames to 'Team'. Use whichever exists.
        team_col_tw = "Team" if "Team" in tw_df.columns else "Squad"
        tw_teams = set(tw_df[team_col_tw].dropna().astype(str).str.strip().unique())
    combined = sorted(perf_teams | sal_teams | tw_teams)
    remaining_old = set(combined) & OLD_ASCII
    print(f"\n{season}: {len(combined)} teams (expected {EXPECTED_COUNTS.get(season)})")
    print("  (a) Old ASCII names remain:" if remaining_old else "  (a) OK: no old ASCII names")
    if remaining_old:
        print("     ", remaining_old)
    print("  Sorted teams:", combined[:15], "..." if len(combined) > 15 else "")

print("Step 4 done.")

## Step 5 — NFC and merge key (maximum merge quality)

**Why:** The same accented character can be stored as NFC (one character) or NFD (base + combining). We normalise all Player names to NFC so they compare equally. Four truncated names get a manual replacement. We define `make_player_key()` for matching only (never stored): it uses NFKD + ASCII so that joins work consistently.

**Steps:** (1) Apply `clean_player_name()` to Player in all 6 files. (2) Verify 0 remaining garbage and print a 20-name sample. (3) Compute match rate (performance player–season vs salary) excluding "2 Teams"/"3 Teams". (4) Export `player_name_audit.csv` with Player_stored, Season, Team, Comp, File, Salary_Matched. (5) Save performance and salary files.

In [ ]:
import re

TRUNCATED_NAME_MAP = {
    "AdriÃ Altimira": "Adrià Altimira",
    "AdriÃ Pedrosa": "Adrià Pedrosa",
    "JosÃ© Luis GayÃ": "José Luis Gayà",
    "Arnau SolÃ": "Arnau Solà",
}

def clean_player_name(s):
    """Full cleaning for stored player name: truncated fix + NFC + strip. Do not remove accents."""
    if not isinstance(s, str):
        return s
    if s in TRUNCATED_NAME_MAP:
        return TRUNCATED_NAME_MAP[s]
    s = s.strip()
    s = unicodedata.normalize("NFC", s)
    return s

def make_player_key(s):
    """Normalized ASCII key for MATCHING ONLY. Never store as player name."""
    if not isinstance(s, str):
        return s
    s = clean_player_name(s)
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii")
    return s.lower().strip()

# --- STEP 2: Apply clean_player_name() to Player in all 6 files ---
PERF_FILE_SHORT = {
    "defenders_combined.xlsx": "Defenders",
    "midfielders_forwards_combined.xlsx": "Midfielders_Forwards",
    "goalkeepers_combined.xlsx": "Goalkeepers",
}

perf_dfs = []
for fname in PERF_FILENAMES:
    path = PERFORMANCE_DIR / fname
    if not path.exists():
        continue
    df = pd.read_excel(path)
    if "Player" in df.columns:
        df["Player"] = df["Player"].apply(clean_player_name)
    perf_dfs.append((fname, df, PERF_FILE_SHORT.get(fname, fname)))

salary_dfs = []
for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists():
        continue
    df = pd.read_csv(path)
    season = next((s for s in SEASONS if s in fname), None)
    if season:
        df["Season"] = season
    if "Player" in df.columns:
        df["Player"] = df["Player"].apply(clean_player_name)
    salary_dfs.append((fname, df))

# --- STEP 3: Verify no truncated names remain; print 20-name sample ---
garbage = ["AdriÃ", "JosÃ©", "Arnau SolÃ"]
found = 0
for fname, df, _ in perf_dfs:
    if "Player" not in df.columns:
        continue
    for _, row in df.iterrows():
        p = str(row.get("Player", ""))
        for g in garbage:
            if g in p:
                found += 1
print(f"Remaining truncated/garbage in performance files: {found} occurrences (expected 0)")

sample_rows = []
for fname, df, short in perf_dfs:
    if "Player" not in df.columns or "Team" not in df.columns or "Season" not in df.columns:
        continue
    sub = df[["Player", "Team", "Season"]].drop_duplicates().dropna(subset=["Player"])
    sub = sub[sub["Player"].astype(str).str.strip() != ""]
    if len(sub) > 0:
        sample_rows.append(sub.sample(min(7, len(sub)), random_state=42))
if sample_rows:
    sample_df = pd.concat(sample_rows, ignore_index=True).drop_duplicates().head(20)
    print("\nSample of 20 player names (Player_name_now_stored | Team | Season):")
    print(sample_df.to_string(index=False))

# --- STEP 4: Build salary_keys and compute match rate ---
master_salary = pd.concat([df for _, df in salary_dfs], ignore_index=True)
salary_keys = set(zip(master_salary["Player"].apply(make_player_key), master_salary["Season"]))

print("\nFile | Season | Total_rows | Excl_multiclub | Matched | Unmatched | Match_%")
print("-" * 70)
for fname, df, short in perf_dfs:
    if "Player" not in df.columns or "Season" not in df.columns or "Team" not in df.columns:
        continue
    for season in SEASONS:
        mask_season = df["Season"].astype(str).str.strip() == season
        sub = df.loc[mask_season]
        total_rows = len(sub)
        multiclub = sub["Team"].astype(str).str.match(r"^\d+\s+Teams$", na=False)
        excl_multiclub = total_rows - multiclub.sum()
        sub_clean = sub[~multiclub]
        matched = sum(1 for _, row in sub_clean.iterrows() if (make_player_key(row["Player"]), season) in salary_keys)
        unmatched = len(sub_clean) - matched
        pct = (100.0 * matched / len(sub_clean)) if len(sub_clean) else 0
        if pct < 80 and len(sub_clean) > 0:
            print(f"  WARNING: Match % below 80% — check name cleaning.")
        print(f"{short:12} | {season} | {total_rows:10} | {int(excl_multiclub):13} | {matched:7} | {unmatched:9} | {pct:5.1f}%")

# --- STEP 5: Export audit ---
audit_rows = []
for fname, df, short in perf_dfs:
    if "Player" not in df.columns:
        continue
    for _, row in df.iterrows():
        p = row["Player"]
        if pd.isna(p) or str(p).strip() == "":
            continue
        season = str(row.get("Season", "")).strip()
        key = (make_player_key(p), season)
        audit_rows.append({
            "Player_stored": p,
            "Season": season,
            "Team": row.get("Team", ""),
            "Comp": row.get("Comp", ""),
            "File": short,
            "Salary_Matched": key in salary_keys,
        })
audit_df = pd.DataFrame(audit_rows).drop_duplicates(subset=["Player_stored", "Season", "File"])
audit_path = OUTPUTS_DIR / "player_name_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"\nExported {audit_path} ({len(audit_df)} rows)")

# --- STEP 6: Save all 3 performance and 3 salary ---
for fname, df, _ in perf_dfs:
    df.to_excel(PERFORMANCE_DIR / fname, index=False)
    print(f"Saved {fname}")
for fname, df in salary_dfs:
    out_df = df.drop(columns=["Season"], axis=1) if "Season" in df.columns else df
    out_df.to_csv(SALARIES_DIR / fname, index=False)
    print(f"Saved {fname}")

print("Step 5 done.")

## Step 6 — Merge salaries and team wages into performance

After all cleaning, we merge player salary data and team wage data into the three performance files. We build **master_salaries** (from the 3 salary CSVs) and **master_team_wages** (from the 3 team wage CSVs), then left-join both into each performance file on `player_key`+Season and Team+Season. Rows with Team "2 Teams" or "3 Teams" keep salary match by player_key+Season but get null team wages (expected). We print a match-rate summary and team wage coverage; if overall match rate is below 85% we stop before saving. Outputs are saved to **Databases/Merged/** (*_merged.xlsx). Intermediate cleaned files in Performance/Salaries/Team_Wages are overwritten in place.

In [ ]:
import re

TRUNCATED = {"AdriÃ Altimira": "Adrià Altimira", "AdriÃ Pedrosa": "Adrià Pedrosa", "JosÃ© Luis GayÃ": "José Luis Gayà", "Arnau SolÃ": "Arnau Solà"}
def clean_player_name(s):
    if not isinstance(s, str): return s
    if s in TRUNCATED: return TRUNCATED[s]
    return unicodedata.normalize("NFC", s.strip())
def make_player_key(s):
    """Normalized key for matching. Strips suffixes (Jr., III, etc.), hyphens, apostrophes; then NFKD+ASCII."""
    if not isinstance(s, str): return s
    s = clean_player_name(s)
    s = re.sub(r"\s+(?:Jr\.?|Sr\.?|III?|IV|II)$", "", s, flags=re.IGNORECASE)
    for c in ["'", "'", "`", "-"]: s = s.replace(c, "")
    s = " ".join(s.split())
    s = unicodedata.normalize("NFKD", s)
    s = s.encode("ascii", "ignore").decode("ascii").lower().strip()
    return s
def extract_eur(s):
    if not isinstance(s, str): return None
    m = re.search(r"€\s*([\d,]+)", s)
    return int(m.group(1).replace(",", "")) if m else None

# STEP 2: master_salaries
sal_list = []
for fname in SALARY_FILENAMES:
    path = SALARIES_DIR / fname
    if not path.exists(): continue
    df = pd.read_csv(path)
    season = next((s for s in SEASONS if s in fname), None)
    if season is None: continue
    df["Season"] = season
    sal_list.append(df)
master_salaries = pd.concat(sal_list, ignore_index=True)
master_salaries["player_key"] = master_salaries["Player"].apply(make_player_key)
master_salaries["Weekly_Wages_EUR"] = master_salaries["Weekly Wages"].apply(extract_eur)
master_salaries["Annual_Wages_EUR"] = master_salaries["Annual Wages"].apply(extract_eur)
keep_sal = ["player_key", "Season", "Weekly Wages", "Annual Wages", "Weekly_Wages_EUR", "Annual_Wages_EUR", "Notes"]
master_salaries = master_salaries[[c for c in keep_sal if c in master_salaries.columns]].copy()
master_salaries = master_salaries.rename(columns={"Notes": "Salary_Notes"})
before_dedup = len(master_salaries)
master_salaries = master_salaries.sort_values("Weekly_Wages_EUR", ascending=False, na_position="last")
master_salaries = master_salaries.drop_duplicates(subset=["player_key", "Season"], keep="first")
if len(master_salaries) < before_dedup:
    print(f"Removed {before_dedup - len(master_salaries)} duplicate player_key+Season from master_salaries.")

# STEP 3: master_team_wages
tw_list = []
for fname in TEAM_WAGES_FILENAMES:
    path = TEAM_WAGES_DIR / fname
    if not path.exists(): continue
    df = pd.read_csv(path)
    season = next((s for s in SEASONS if s in fname), None)
    if season is None: continue
    df["Season"] = season
    tw_list.append(df)
master_team_wages = pd.concat(tw_list, ignore_index=True)
master_team_wages = master_team_wages.rename(columns={"Squad": "Team", "Weekly Wages": "Team_Weekly_Wages", "Annual Wages": "Team_Annual_Wages", "% Estimated": "Wages_Pct_Estimated", "# Pl": "Squad_Size"})
if "Rk" in master_team_wages.columns: master_team_wages = master_team_wages.drop(columns=["Rk"])
master_team_wages["Team_Weekly_Wages_EUR"] = master_team_wages["Team_Weekly_Wages"].apply(extract_eur)
master_team_wages["Team_Annual_Wages_EUR"] = master_team_wages["Team_Annual_Wages"].apply(extract_eur)
keep_tw = ["Team", "Season", "Team_Weekly_Wages", "Team_Annual_Wages", "Team_Weekly_Wages_EUR", "Team_Annual_Wages_EUR", "Wages_Pct_Estimated", "Squad_Size"]
master_team_wages = master_team_wages[[c for c in keep_tw if c in master_team_wages.columns]]

# STEP 4: Merge into each performance file
DESIRED_ORDER = ["Rk", "Player", "Season", "Age", "Nation", "Team", "Comp", "Pos", "MP", "Min", "90s", "Starts", "Subs", "unSub", "Gls", "Ast", "G+A", "G-PK", "PK", "PKatt", "PKm", "Sh", "Sh_Standard", "G/Sh", "G/SoT", "SoT", "SoT%", "TklW", "Int", "Int_Def", "Tkl+Int", "Mn/MP", "Mn/MP_Detail", "Min%", "Mn/Start", "Compl", "Mn/Sub", "PPM", "PPM_Team", "onG", "onGA", "+/-", "On-Off", "CrdY", "CrdY_Detail", "CrdR", "2CrdY", "Fls", "Fld", "Off", "PKwon", "PKcon", "OG", "GA", "SoTA", "Saves", "Save%", "W", "D", "L_Detail", "CS", "CS%", "PKvs", "GA_Summary", "PKvs_Summary", "PKA", "PKsv", "PKm_GK", "Save%_PK", "OG_Detail", "Off_Detail", "Weekly Wages", "Annual Wages", "Weekly_Wages_EUR", "Annual_Wages_EUR", "Salary_Notes", "Salary_Matched", "Team_Weekly_Wages", "Team_Annual_Wages", "Team_Weekly_Wages_EUR", "Team_Annual_Wages_EUR", "Wages_Pct_Estimated", "Squad_Size"]
# Output filenames for merged tables (saved to Databases/Merged/)
OUTPUT_NAMES = {
    "defenders_combined.xlsx": "defenders_merged.xlsx",
    "midfielders_forwards_combined.xlsx": "midfielders_forwards_merged.xlsx",
    "goalkeepers_combined.xlsx": "goalkeepers_merged.xlsx",
}
FILE_SHORT = {
    "defenders_combined.xlsx": "Defenders",
    "midfielders_forwards_combined.xlsx": "Midfielders_Forwards",
    "goalkeepers_combined.xlsx": "Goalkeepers",
}

merged_list = []
for fname in PERF_FILENAMES:
    path = PERFORMANCE_DIR / fname
    if not path.exists(): continue
    df = pd.read_excel(path)
    df["player_key"] = df["Player"].apply(make_player_key)
    df = df.merge(master_salaries, on=["player_key", "Season"], how="left")
    df = df.merge(master_team_wages, on=["Team", "Season"], how="left")
    df["Salary_Matched"] = df["Weekly_Wages_EUR"].notna()
    df = df.drop(columns=["player_key"])
    ordered = [c for c in DESIRED_ORDER if c in df.columns]
    remaining = [c for c in df.columns if c not in ordered]
    df = df[ordered + remaining]
    merged_list.append((FILE_SHORT.get(fname, fname), fname, df))

# STEP 5: Match rate summary
overall_matched = overall_total = 0
for short, fname, df in merged_list:
    print(f"\nFILE: {short}")
    print("┌─────────────┬────────────┬──────────┬────────────┬──────────┐")
    print("│ Season      │ Total rows │ Matched  │ Unmatched  │ Match %  │")
    print("├─────────────┼────────────┼──────────┼────────────┼──────────┤")
    file_total = file_matched = 0
    for season in SEASONS:
        sub = df[df["Season"].astype(str).str.strip() == season]
        total, matched = len(sub), int(sub["Salary_Matched"].sum())
        unmatched = total - matched
        pct = (100.0 * matched / total) if total else 0
        file_total += total; file_matched += matched
        print(f"│ {season:<11} │ {total:>10} │ {matched:>8} │ {unmatched:>10} │ {pct:>6.1f}% │")
    pct_tot = (100.0 * file_matched / file_total) if file_total else 0
    print(f"│ {'TOTAL':<11} │ {file_total:>10} │ {file_matched:>8} │ {file_total - file_matched:>10} │ {pct_tot:>6.1f}% │")
    print("└─────────────┴────────────┴──────────┴────────────┴──────────┘")
    tw_ok = df["Team_Weekly_Wages_EUR"].notna().sum()
    print(f"Team wage coverage: {100.0 * tw_ok / len(df):.1f}%")
    overall_total += file_total; overall_matched += file_matched

overall_pct = (100.0 * overall_matched / overall_total) if overall_total else 0
if overall_pct < 85:
    print(f"\nOverall match rate {overall_pct:.1f}% < 85%. Stopping — do not save.")
else:
    for short, fname, df in merged_list:
        out_name = OUTPUT_NAMES.get(fname, fname.replace(".xlsx", "_merged.xlsx"))
        out_path = MERGED_DIR / out_name
        df.to_excel(out_path, index=False, engine="openpyxl")
        print(f"Saved {out_path}")
    print("Step 6 done.")

## Step 7 — Cleanup and split (Merged → Analysis_Ready + Unmatched)

After the merge (Step 6), the files in `data/processed/merged/` contain all rows (matched and unmatched) and still have team wage columns and the Rk column. This step:

1. **Remove exact duplicate rows** — Only rows that are identical in every column are dropped.
2. **Drop all-null columns** — Any column with no non-null values is removed.
3. **Drop Rk and team wage columns** — Rk is the source row index; team wage columns are removed from the analysis tables.
4. **Overwrite `data/processed/merged/`** — The three merged files are updated in place with the cleaned columns and de-duplicated rows.
5. **Write `data/processed/feature_engineered/`** — One file per position: only rows where **Salary_Matched == True** and **Min ≥ 90**. Then **Salary_Notes** and **Salary_Matched** are dropped. These are the inputs for feature engineering.
6. **Write `data/processed/unmatched/`** — One file per position: rows where **Salary_Matched == False**. Wage columns and **L** are dropped before saving.

In [ ]:
# Columns to remove from the final tables (Rk = source row index; team wage cols not needed for player-level analysis)
COLS_TO_DROP = [
    "Rk",
    "Team_Weekly_Wages",
    "Team_Annual_Wages",
    "Team_Weekly_Wages_EUR",
    "Team_Annual_Wages_EUR",
    "Wages_Pct_Estimated",
    "Squad_Size",
]

MERGED_FILES = ["defenders_merged.xlsx", "midfielders_forwards_merged.xlsx", "goalkeepers_merged.xlsx"]

for fname in MERGED_FILES:
    path = MERGED_DIR / fname
    if not path.exists():
        print(f"Skip (not found): {path}")
        continue
    df = pd.read_excel(path, engine="openpyxl")

    # Remove only exact duplicate rows (identical in every column)
    n_before = len(df)
    df = df.drop_duplicates(keep="first")
    n_removed = n_before - len(df)
    if n_removed:
        print(f"{fname}: removed {n_removed} exact duplicate rows (identical column-by-column).")

    # Drop columns that are 100% null
    null_cols = [c for c in df.columns if df[c].isna().all()]
    if null_cols:
        print(f"{fname}: dropping all-null columns: {null_cols}")
        df = df.drop(columns=null_cols)

    # Drop Rk and team wage columns
    to_drop = [c for c in COLS_TO_DROP if c in df.columns]
    if to_drop:
        df = df.drop(columns=to_drop)

    # Overwrite Merged (all rows, cleaned columns)
    df.to_excel(path, index=False, engine="openpyxl")
    print(f"Updated {path.name} (all rows, cleaned columns).")

    if "Salary_Matched" not in df.columns:
        print(f"  No Salary_Matched column; skip Analysis_Ready/Unmatched.")
        continue

    stem = fname.replace("_merged.xlsx", "")

    # --- Analysis_Ready: only Salary_Matched==True; Min>=90; then drop Salary_Notes and Salary_Matched ---
    df_def = df[df["Salary_Matched"] == True].copy()
    # Minimum minutes filter: below 90 min, per-90 metrics are degenerate (see feature_engineering prompt / methodology).
    df_def = df_def[pd.to_numeric(df_def["Min"], errors="coerce") >= 90].copy()
    print(f"  After Min>=90 filter: {len(df_def)} rows remain")
    for c in ["Salary_Notes", "Salary_Matched"]:
        if c in df_def.columns:
            df_def = df_def.drop(columns=[c])
    out_ready = ANALYSIS_READY_DIR / f"{stem}_analysis_ready.xlsx"
    df_def.to_excel(out_ready, index=False, engine="openpyxl")
    print(f"  Saved {out_ready.name} ({len(df_def)} rows, Salary_Matched=True).")

    # --- Unmatched: only Salary_Matched==False; then drop all wage columns and L ---
    df_unm = df[df["Salary_Matched"] == False].copy()
    # Wage columns dropped (no salary data for unmatched). 'L' is Defenders-only; guard skips it for other files.
    cols_unm_drop = ["Weekly Wages", "Annual Wages", "Weekly_Wages_EUR", "Annual_Wages_EUR", "Salary_Notes", "Salary_Matched", "L"]
    cols_unm_drop = [c for c in cols_unm_drop if c in df_unm.columns]
    if cols_unm_drop:
        df_unm = df_unm.drop(columns=cols_unm_drop)
    out_unm = UNMATCHED_DIR / f"{stem}_unmatched.xlsx"
    df_unm.to_excel(out_unm, index=False, engine="openpyxl")
    print(f"  Saved {out_unm.name} ({len(df_unm)} rows, Salary_Matched=False).")

print("Step 7 done.")

## Step 8 — Pipeline validation report

After Step 7, the three Analysis_Ready files are written. This cell runs an automated quality gate: 10 checks on each file (required columns present, no forbidden columns, wage null rate zero, wage range sane, valid Comp/Season/Nation, no Player+Team+Season duplicates, Min≥90). Each check reports PASS or FAIL so you can confirm the pipeline outputs are valid before analysis.

In [ ]:
print("=" * 60)
print("STEP 8 — PIPELINE VALIDATION REPORT")
print("=" * 60)

EXPECTED_COLS = ["Player", "Season", "Age", "Nation", "Team", "Comp", "Pos", "MP", "Min", "Weekly_Wages_EUR", "Annual_Wages_EUR"]
FORBIDDEN_COLS = ["Rk", "Salary_Matched", "Salary_Notes", "Team_Weekly_Wages_EUR", "Squad"]
VALID_COMPS = {"Bundesliga", "La Liga", "Ligue 1", "Premier League", "Serie A", "2 Comps", "3 Comps"}
VALID_SEASONS = {"2022-2023", "2023-2024", "2024-2025"}

for fname in ["defenders_analysis_ready.xlsx", "midfielders_forwards_analysis_ready.xlsx", "goalkeepers_analysis_ready.xlsx"]:
    path = ANALYSIS_READY_DIR / fname
    if not path.exists():
        print(f"FAIL {fname} not found")
        continue
    df = pd.read_excel(path, engine="openpyxl")
    print(f"\n--- {fname} ({len(df)} rows) ---")
    missing = [c for c in EXPECTED_COLS if c not in df.columns]
    print(f" [PASS] Required columns: all present" if not missing else f" [FAIL] Required columns missing: {missing}")
    present = [c for c in FORBIDDEN_COLS if c in df.columns]
    print(f" [PASS] No forbidden cols" if not present else f" [FAIL] Forbidden cols present: {present}")
    null_w = df["Weekly_Wages_EUR"].isna().mean() if "Weekly_Wages_EUR" in df.columns else 1.0
    print(f" [PASS] Wage null rate: 0%" if null_w == 0 else f" [FAIL] Wage null rate: {null_w*100:.1f}%")
    wages = pd.to_numeric(df["Weekly_Wages_EUR"], errors="coerce").dropna() if "Weekly_Wages_EUR" in df.columns else pd.Series(dtype=float)
    if len(wages) > 0:
        sane = ((wages >= 500) & (wages <= 2_000_000)).all()
        print(f" [PASS] Wage range sane" if sane else f" [FAIL] Wage range: min={wages.min():,.0f} max={wages.max():,.0f}")
        neg = (wages < 0).sum()
        print(f" [PASS] No negative wages" if neg == 0 else f" [FAIL] Negative wages: {neg}")
    bad_comp = set(df["Comp"].dropna().astype(str).str.strip().unique()) - VALID_COMPS if "Comp" in df.columns else set()
    print(f" [PASS] Comp values valid" if not bad_comp else f" [FAIL] Comp values: {bad_comp}")
    found_s = set(df["Season"].dropna().astype(str).str.strip().unique()) if "Season" in df.columns else set()
    print(f" [PASS] Seasons: {sorted(found_s)}" if VALID_SEASONS <= found_s else f" [FAIL] Seasons: {sorted(found_s)}")
    if "Nation" in df.columns:
        bad_nation = df["Nation"].dropna().astype(str).str.strip()
        bad_nation = bad_nation[bad_nation.str.len() != 3]
        print(f" [PASS] Nation 3-letter" if len(bad_nation) == 0 else f" [FAIL] Nation non-3-letter: {len(bad_nation)}")
    dupes = df.duplicated(subset=["Player", "Team", "Season"]).sum() if all(c in df.columns for c in ["Player", "Team", "Season"]) else 0
    print(f" [PASS] No Player+Team+Season dupes" if dupes == 0 else f" [FAIL] Duplicates: {dupes}")
    low_min = (pd.to_numeric(df["Min"], errors="coerce") < 90).sum() if "Min" in df.columns else 0
    print(f" [PASS] Min>=90" if low_min == 0 else f" [WARN] Rows with Min<90: {low_min}")

print("\n" + "=" * 60)
print("Validation complete.")

### How to improve the match rate further

- **Already applied:** In `make_player_key()` we strip suffixes (Jr., III, II, IV), apostrophes and hyphens, and normalise to an ASCII key, so cases like "Jean-Pierre" / "Jean Pierre" or "O'Brien" / "OBrien" match.
- **Data limitation:** Many unmatched rows are simply **not present** in the salary CSVs (FBRef does not include them, or they are at another club that season). A manual check shows most unmatched players have no row in the salary file.
- **Optional improvements:**
  1. **Fuzzy matching:** For rows still unmatched, search the same-season salary file for the closest name (e.g. with `rapidfuzz`) and assign salary if similarity exceeds a threshold (e.g. 92%). Risk: some false positives. Requires `pip install rapidfuzz`.
  2. **Manual mapping:** Add a dictionary of known same-player names (e.g. "Gavi" → full name in salaries) and apply it before the merge.
  3. **Leave as is:** A ~86% match rate is reasonable if the salary source does not cover all performance players.

### Rule for wage-efficiency analysis (TFG)

To compare players across teams using the ratio **player’s % contribution to the team** vs **player’s % of team wage bill**, use only rows where the player has **a single team** in that season:

- **Exclude** rows with `Team = "2 Teams"` or `Team = "3 Teams"` (players who changed club mid-season; contribution is aggregated and cannot be assigned to one team).
- In the TFG methodology and limitations: state that team-relative efficiency analysis is performed on players with one team in the season.

Optional: for multi-team players you can use an **absolute** metric (total contribution / salary) in a separate section, without comparing to team wage spend.

In [ ]:
# DataFrames for wage-efficiency analysis: only players with a single team in the season
MULTITEAM = ["2 Teams", "3 Teams"]

# If you just ran Step 6, merged_list exists; otherwise load from Databases/Merged/
try:
    efficiency_dfs = []
    for short, fname, df in merged_list:
        mask_one_team = ~df["Team"].astype(str).str.strip().isin(MULTITEAM)
        df_one = df[mask_one_team].copy()
        df_one["File"] = short
        efficiency_dfs.append(df_one)
        print(f"{short}: {len(df_one)} rows with one team (excl. {(~mask_one_team).sum()} multi-team)")
    df_one_team = pd.concat(efficiency_dfs, ignore_index=True)
    print(f"\nTotal rows for efficiency analysis (one team): {len(df_one_team)}")
except NameError:
    efficiency_dfs = []
    MERGED_NAMES = {"Defenders": "defenders_merged.xlsx", "Midfielders_Forwards": "midfielders_forwards_merged.xlsx", "Goalkeepers": "goalkeepers_merged.xlsx"}
    for short, fname in MERGED_NAMES.items():
        path = MERGED_DIR / fname
        if not path.exists():
            continue
        df = pd.read_excel(path)
        mask_one_team = ~df["Team"].astype(str).str.strip().isin(MULTITEAM)
        df_one = df[mask_one_team].copy()
        df_one["File"] = short
        efficiency_dfs.append(df_one)
        print(f"{short}: {len(df_one)} rows (excl. {(~mask_one_team).sum()} multi-team)")
    df_one_team = pd.concat(efficiency_dfs, ignore_index=True) if efficiency_dfs else None
    if df_one_team is not None:
        print(f"\nTotal rows for efficiency analysis: {len(df_one_team)}")

# Optional: save a single CSV/Excel for TFG analysis (one-team only)
# df_one_team.to_excel(OUTPUTS_DIR / "efficiency_analysis_one_team.xlsx", index=False, engine="openpyxl")
# df_one_team.to_csv(OUTPUTS_DIR / "efficiency_analysis_one_team.csv", index=False)